# Dual-Band Sweep + ORx AGC - ADRV9026

Declare a **sweep plan** (`SWEEP` with per-block `zip` or `grid`), preview with
`summarize_sweep_plan`, then at **every point** auto-level each ORx independently
and capture per band.

## About the ORx "AGC"

There is no trustworthy hardware AGC for the ORx here, so we auto-level in software
on the **captured-IQ peak** with a **railed-sample clip veto** (`RxDecPowerGet`
compresses the range and `RxGainGet` returns 0, so the gain index is tracked in
software). At every sweep point `autolevel_orx` starts at the gain floor and trims
the ORx gain index (clamped to the valid **185..255** window) until the captured
peak lands in the asymmetric band `[target - tol_down, target + tol_up]`
(defaults -1.0 +0.3/-0.6 dBFS) with `railed == 0`.

If the TX is too strong to fit the target even at the gain floor (185) it stops as a
**fatal** condition (reduce TX power); if the signal is too weak to reach the band
even at max gain (255) it **accepts 255** as the best achievable (`at_max_gain`) --
watch the `leveled` column and the red X's on the plot.

## 1. Parameters (edit me)

In [ ]:
from adrvtrx import TxChannel, RxChannel

PROFILES = {
    "98_linksharing":  "ADRV9025Init_StdUseCase98_LinkSharing.profile",
    "102_linksharing": "ADRV9025Init_StdUseCase102_LinkSharing.profile",
    "14_linksharing":  "ADRV9025Init_StdUseCase14_LinkSharing.profile",
    "50_linksharing":  "ADRV9025Init_StdUseCase50_LinkSharing.profile",
}
PROFILE = "98_linksharing"

INPUT_DIR = "C:/Users/ohammi/OneDrive - aus.edu/DualBand/input_100"
BANDS = [
    {"name": "band1", "tx": TxChannel.TX2, "orx": RxChannel.ORX2,
     "signal": f"{INPUT_DIR}/Signal1.txt"},
    {"name": "band2", "tx": TxChannel.TX3, "orx": RxChannel.ORX3,
     "signal": f"{INPUT_DIR}/Signal2.txt"},
]

# Each block: mode "zip" (paired by index) or "grid" (combinatorial). Blocks multiply.
SWEEP = {
    "power_db": {"mode": "grid", "shared": [20.0, 15.0, 10.0, 5.0, 0.0]},
    # "freq": {"mode": "zip", "lo1_hz": [1.0e9, 1.1e9], "lo2_hz": [0.9e9, 1.0e9]},
    # "power_db": {"mode": "grid", "shared": [13, 14, 15]},
    # "signals": {"mode": "grid", "band1": [f"{INPUT_DIR}/Signal1a.txt", ...], "band2": "..."},
}

USE_AGC          = True
ORX_TARGET_DBFS  = -1.0
ORX_TOL_UP_DB    = 0.3
ORX_TOL_DOWN_DB  = 0.6
ORX_GAIN_INDEX   = 240
CAPTURE_MS       = 0.05
OUTPUT_OVERSAMPLE = 2
SAVE_DIR          = None


## 2. Imports, config, profile

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.capture import capture
from adrvtrx.gain import clip_report, autolevel_orx, ORX_GAIN_MIN, ORX_GAIN_MAX
from adrvtrx.sweep_plan import (
    flatten_point, format_point_label, run_planned_sweep,
    summarize_sweep_plan, sweep_defaults_from_config,
)
from adrvtrx.align import estimate_delay, match_corr

cfg = load_config()
cfg.profile_name = PROFILES.get(PROFILE, PROFILE)
SWEEP_DEFAULTS = sweep_defaults_from_config(cfg, BANDS)
info = read_profile(cfg.profile_path)
print(f"LO1 = {cfg.lo.lo1_hz/1e6:.1f} MHz | LO2 = {cfg.lo.lo2_hz/1e6:.1f} MHz")
print("Profile:", cfg.profile_path.name)
print(f"ORx auto-level (AGC); gain window {ORX_GAIN_MIN}..{ORX_GAIN_MAX}")
print(summarize_sweep_plan(BANDS, SWEEP, SWEEP_DEFAULTS))

## 3. Signal-path summary - sample rates & bit depth

In [ ]:
def _fs(bits):
    return (1 << (bits - 1)) - 1

print(f"{'path':<6}{'rate (MSPS)':>14}{'bits (Np)':>12}{'full scale':>12}")
print(f"{'TX':<6}{info.tx_rate_khz/1000:>14.3f}{info.tx_bits:>12}{_fs(info.tx_bits):>12}")
print(f"{'Rx':<6}{info.rx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")
print(f"{'ORx':<6}{info.orx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")

## 4. Signals

Waveforms load per sweep point via `run_planned_sweep` (cached by path). Equal length required.
Per-band ORx capture is sequential (hardware constraint — see `development_debugging.md`).

In [ ]:
# (no preload — see run_planned_sweep in section 7)

## 5. Connect, program, verify

In [ ]:
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()
for k, v in verify_status(radio).items():
    print(f"{k}: {v}")

## 6. Build the sweep axis + per-point action (auto-level each ORx, capture both)

In [ ]:
def measure(ch):
    cap = capture(radio, int(ch), CAPTURE_MS, bits=info.rx_bits).channels[ch]
    rep = clip_report(cap.i, cap.q, info.rx_bits)
    return rep.peak_dbfs, rep.railed_samples

def action(point, waves):
    flat = flatten_point(point)
    label = format_point_label(point)
    row = {**flat}
    for b in BANDS:
        if USE_AGC:
            lr = autolevel_orx(
                lambda g, ch=b["orx"]: radio.set_rx_gain(ch, g),
                lambda ch=b["orx"]: measure(ch),
                target_dbfs=ORX_TARGET_DBFS, tol_up_db=ORX_TOL_UP_DB, tol_down_db=ORX_TOL_DOWN_DB,
            )
            row[f"{b['name']}_gain"] = lr.final_gain_index
            row[f"{b['name']}_leveled"] = lr.converged
        else:
            radio.set_rx_gain(b["orx"], ORX_GAIN_INDEX)
            row[f"{b['name']}_gain"] = ORX_GAIN_INDEX
            row[f"{b['name']}_leveled"] = True
    ref_len = len(next(iter(waves.values())))
    out_ms = OUTPUT_OVERSAMPLE * ref_len / info.orx_rate_hz * 1e3
    if SAVE_DIR:
        from pathlib import Path
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
    for b in BANDS:
        nm = b["name"]
        ref = waves[b["tx"]]
        out = capture(radio, int(b["orx"]), out_ms, bits=info.rx_bits).channels[b["orx"]]
        rep = clip_report(out.i, out.q, info.rx_bits)
        y = out.i.astype(float) + 1j * out.q.astype(float)
        delay = estimate_delay(ref, y, info.orx_rate_hz)
        corr = match_corr(ref, y, info.orx_rate_hz)
        saved = ""
        if SAVE_DIR:
            saved = f"{SAVE_DIR}/{label}_{nm}.txt"
            out.save(saved)
        row[f"{nm}_peak"] = rep.peak_dbfs
        row[f"{nm}_delay_ns"] = delay / info.orx_rate_hz * 1e9
        row[f"{nm}_corr"] = corr
        print(f"  {label} {nm}: gain {row[nm+'_gain']}, peak {rep.peak_dbfs:+.1f} dBFS, "
              f"delay {row[nm+'_delay_ns']:.1f} ns, corr {corr:.3f}" + (f" -> {saved}" if saved else ""))
    return row

## 7. Run the sweep

In [ ]:
records = run_planned_sweep(
    radio, BANDS, SWEEP, action, defaults=SWEEP_DEFAULTS, tx_bits=info.tx_bits,
)
radio.disable_tx()

def _xkey(records):
    for k in records[0]:
        if k.endswith("_power_db") or k.endswith("_hz"):
            if len({r[k] for r in records}) > 1:
                return k
    return None

XKEY = _xkey(records) or "point"
XLABEL = XKEY.replace("_", " ")
hdr = f"\n{'#':>4} {XLABEL:>14}"
for b in BANDS:
    nm = b["name"]
    hdr += f" {nm+'_gain':>10}{nm+'_peak':>9}{nm+'_dly_ns':>10}{nm+'_corr':>7}"
print(hdr)
for i, r in enumerate(records):
    x = r.get(XKEY, i)
    line = f"{i:>4} {str(x):>14}"
    for b in BANDS:
        nm = b["name"]
        line += (f" {r[nm+'_gain']:>10}{r[nm+'_peak']:>9.1f}"
                 f"{r[nm+'_delay_ns']:>10.1f}{r[nm+'_corr']:>7.3f}")
    print(line)

## 8. Plot - per-band gain + leveled capture vs sweep

In [ ]:
if XKEY in records[0]:
    x = np.array([r[XKEY] for r in records], float)
else:
    x = np.arange(len(records))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for b in BANDS:
    ax[0].plot(x, [r[f"{b['name']}_gain"] for r in records], "o-", label=b["name"])
    ax[1].plot(x, [r[f"{b['name']}_peak"] for r in records], "o-", label=b["name"])
ax[1].axhline(ORX_TARGET_DBFS, ls="--", c="k", lw=1, label="target")
ax[0].set_title("settled ORx gain index"); ax[0].set_xlabel(XLABEL)
ax[0].set_ylabel("gain index"); ax[0].legend(); ax[0].grid(True)
ax[1].set_title("captured peak after AGC"); ax[1].set_xlabel(XLABEL)
ax[1].set_ylabel("dBFS"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

## 9. Safe-state & disconnect

In [ ]:
radio.safe_state()
radio.disconnect()
print("TX safe, board disconnected")